<a href="https://colab.research.google.com/github/SriSharanya-617/GPT/blob/main/GPT_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1: Data Preparation

Objective
Prepare text data for GPT training.

. Load dataset
. Convert to lowercase
. Remove unwanted symbols
· Tokenize text
. Build vocabulary
. Create input-output sequences

Example: Input: 'the cat chased' Target: 'the'

In [2]:
texts = [
"The cat chased the mouse.",
"The dog barked loudly.",
"Machine learning is powerful.",
"Deep learning uses neural networks."
]

# Import necessary library
import re
from tensorflow.keras.preprocessing.text import Tokenizer

texts=[re.sub(r'[^a-zA-Z\s]','',t.lower()) for t in texts]

tokenizer=Tokenizer()
tokenizer.fit_on_texts(texts)

word_index=tokenizer.word_index
vocab_size=len(word_index)+1

sequences=[]

for text in texts:
    token_list=tokenizer.texts_to_sequences([text])[0]
    for i in range(1,len(token_list)):
        sequences.append(token_list[:i+1])

Task 2 :Token Embeddings  

In [3]:
import tensorflow as tf
embedding_dim=32
embedding_layer=tf.keras.layers.Embedding(vocab_size,embedding_dim)

sample_input=tf.constant([1,2,3])
embedded=embedding_layer(sample_input)

print("Embeddings",embedded)
print("Embedding Shape",embedded.shape)

Embeddings tf.Tensor(
[[ 0.00742523  0.04134958  0.00298058 -0.04673374  0.00409029  0.0050082
   0.02157212  0.03651507  0.04693476 -0.02665755 -0.04910611 -0.01993526
   0.0424895  -0.00877697 -0.04840529  0.0407165   0.01542665  0.01780436
   0.0369092  -0.0260061  -0.01160258  0.03940848  0.01437858 -0.03324552
   0.03682274 -0.01668491 -0.03471494 -0.02698451 -0.03561019 -0.02964681
  -0.03573763  0.02999176]
 [-0.02268788 -0.04788334 -0.04454286 -0.01856202  0.0060931  -0.04360551
  -0.02048051  0.04839455  0.04643359  0.00730031 -0.01813631  0.00919838
  -0.00850803 -0.01421318  0.01119504  0.02402959  0.03059442 -0.04966477
   0.00318457 -0.00797384  0.04072514 -0.00226174 -0.00563102 -0.04743461
   0.03830298 -0.0333017  -0.04212696  0.00636616  0.0398841  -0.04547037
  -0.01812945  0.01780475]
 [-0.01884254 -0.00089625  0.02993547 -0.03354289 -0.0070773  -0.03970578
   0.01442143 -0.01163659 -0.0008531  -0.00963557  0.02340282  0.04600898
  -0.0129826  -0.01580392  0.01344395

Task 3 : Positional Encoding

In [4]:
import numpy as np

def positional_encoding(max_len, d_model):
    pos = np.arange(max_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
    angles = pos * angle_rates

    pe = np.zeros((max_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])

    return pe

pe = positional_encoding(10, 32)
print(pe.shape)

(10, 32)


Task - 4

In [5]:
import tensorflow as tf

seq_len=5

mask=1-tf.linalg.band_part(tf.ones((seq_len,seq_len)),-1,0)
print("Attention Mask",mask.numpy())

Attention Mask [[0. 1. 1. 1. 1.]
 [0. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]]


Task 5 : Multihead attention
head1-->Grammar head2-->context head3-->long dependencies head4-->semantics

In [6]:
mha=tf.keras.layers.MultiHeadAttention(
    num_heads=4,
    key_dim=32
)
x=tf.random.normal((2,5,32))

attn_output=mha(x,x,attention_mask=mask)
print("Attention Output",attn_output.shape)

Attention Output (2, 5, 32)


Task 6: GPT Decoder Block

Objective

Build a Decoder Block from:

1. Masked Multi Head Attention
2. Add & Normalize
3. Feed Forward Network
4.add and normalize

In [8]:
import tensorflow as tf

class DecoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff):
        super().__init__()

        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model
        )

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_model)
        ])

        self.norm1 = tf.keras.layers.LayerNormalization()
        self.norm2 = tf.keras.layers.LayerNormalization()

    def call(self, x):
        attn = self.mha(x, x, use_causal_mask=True)
        x = self.norm1(x + attn)

        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)

        return x

Task 7: GPT Model

Objective
Build GPT Architecture

Input - Embedding - Positional Encoding - Decoder Blocks -+ Linear - Softmax

In [16]:
inputs = tf.keras.Input(shape=(None,))

x = tf.keras.layers.Embedding(vocab_size, 32)(inputs)

decoder = DecoderBlock(32, 4, 64)
x = decoder(x)   # second decoder block

outputs = tf.keras.layers.Dense(vocab_size, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, None, 32)       │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_block_1 (DecoderBlock)  │ (None, None, 32)       │        21,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, None, 16)       │           528 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,160 (86.56 KB)

 Trainable params: 22,160 (86.56 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

X = []
y = []

for seq in sequences:
    X.append(seq[:-1])
    y.append(seq[-1])

X = pad_sequences(X, padding='pre')
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (14, 4)
y shape: (14,)


In [20]:
print(model.output_shape)

(None, None, 16)


In [21]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [22]:
inputs = tf.keras.Input(shape=(None,))

x = tf.keras.layers.Embedding(vocab_size, 32)(inputs)

x = DecoderBlock(32, 4, 64)(x)
x = DecoderBlock(32, 4, 64)(x)

# Keep only the last token representation
x = tf.keras.layers.Lambda(lambda t: t[:, -1, :])(x)

outputs = tf.keras.layers.Dense(vocab_size, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(model.output_shape)

(None, 16)


In [23]:
history = model.fit(
    X,
    y,
    epochs=200,
    batch_size=2,
    verbose=1
)

Epoch 1/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.0000e+00 - loss: 3.5426
Epoch 2/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2143 - loss: 2.3335     
Epoch 3/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5714 - loss: 1.7978 
Epoch 4/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6429 - loss: 1.3425 
Epoch 5/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7143 - loss: 0.9818 
Epoch 6/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8571 - loss: 0.7179 
Epoch 7/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7857 - loss: 0.5612 
Epoch 8/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9286 - loss: 0.4927 
Epoch 9/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8571 - loss: 0.4275 
Epoch 10/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8571 - loss: 0.3631 
Epoch 11/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9286 - loss: 0.3147 
Epoch 12/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.857

In [24]:
loss, acc = model.evaluate(X, y)


print("Loss:", loss)
print("Accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 457ms/step - accuracy: 0.9286 - loss: 0.1037
Loss: 0.10371670871973038
Accuracy: 0.9285714030265808


In [25]:
sentence = "The dog barked"

tokens = tokenizer.texts_to_sequences([sentence])[0]

input_tokens = np.array([tokens])

predictions = model.predict(input_tokens, verbose=0)

predicted_token_id = np.argmax(predictions[0])

reverse_word_index = {
    v: k for k, v in tokenizer.word_index.items()
}

predicted_word = reverse_word_index.get(predicted_token_id, "<unk>")

print("Predicted Next Word:", predicted_word)

Predicted Next Word: loudly
